# NetFlow v2 Two-Stage Preprocess

운영 오탐을 줄이기 위해 `Benign vs Attack` 1단계와 `Attack Type` 2단계를 분리한 산출물을 만든다.

In [1]:
from pathlib import Path
import json
import pickle

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'training').exists():
    PROJECT_ROOT = Path.cwd().parents[1]

BASE_FEATURES = [
    'L4_SRC_PORT', 'L4_DST_PORT', 'PROTOCOL', 'L7_PROTO', 'IN_BYTES', 'OUT_BYTES',
    'IN_PKTS', 'OUT_PKTS', 'TCP_FLAGS', 'FLOW_DURATION_MILLISECONDS',
]
DERIVED_FEATURES = [
    'TOTAL_BYTES', 'TOTAL_PKTS', 'BYTES_PER_PACKET', 'BYTES_PER_SECOND',
    'PACKETS_PER_SECOND', 'IN_OUT_BYTES_RATIO', 'IN_OUT_PKTS_RATIO',
]
FEATURE_NAMES = [*BASE_FEATURES, *DERIVED_FEATURES]
ATTACK_LABEL_MAPPING = {
    'Bot': 0,
    'Brute Force': 1,
    'DDoS': 2,
    'DoS': 3,
    'Infiltration': 4,
    'SQL Injection': 5,
}
BINARY_LABEL_MAPPING = {'Benign': 0, 'Attack': 1}

DATASET_PATH = PROJECT_ROOT / 'training/data/88a47ba2ab64258e_MOHANAD_A4706/data/NF-CSE-CIC-IDS2018.csv'
OUTPUT_DIR = PROJECT_ROOT / 'training/data/processed_netflow_v2_twostage'
BENIGN_MAX_ROWS = 1_000_000
ATTACK_MAX_ROWS_PER_CLASS = 200_000
TEST_SIZE = 0.15
VAL_SIZE = 0.15
RANDOM_STATE = 42
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def normalize_attack(value: str) -> str:
    attack = str(value).strip()
    lowered = attack.lower()
    if lowered == 'benign':
        return 'Benign'
    if 'bot' in lowered:
        return 'Bot'
    if 'brute' in lowered or lowered in {'ftp-bruteforce', 'ssh-bruteforce'}:
        return 'Brute Force'
    if 'ddos' in lowered:
        return 'DDoS'
    if lowered.startswith('dos'):
        return 'DoS'
    if 'infilteration' in lowered or 'infiltration' in lowered:
        return 'Infiltration'
    if 'sql injection' in lowered:
        return 'SQL Injection'
    raise ValueError(f'지원하지 않는 Attack 라벨입니다: {value!r}')

def add_derived_features(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    duration_ms = result['FLOW_DURATION_MILLISECONDS'].clip(lower=1)
    duration_seconds = duration_ms / 1000.0
    result['TOTAL_BYTES'] = result['IN_BYTES'] + result['OUT_BYTES']
    result['TOTAL_PKTS'] = result['IN_PKTS'] + result['OUT_PKTS']
    result['BYTES_PER_PACKET'] = result['TOTAL_BYTES'] / result['TOTAL_PKTS'].replace(0, np.nan)
    result['BYTES_PER_SECOND'] = result['TOTAL_BYTES'] / duration_seconds
    result['PACKETS_PER_SECOND'] = result['TOTAL_PKTS'] / duration_seconds
    result['IN_OUT_BYTES_RATIO'] = result['IN_BYTES'] / result['OUT_BYTES'].replace(0, np.nan)
    result['IN_OUT_PKTS_RATIO'] = result['IN_PKTS'] / result['OUT_PKTS'].replace(0, np.nan)
    result.replace([np.inf, -np.inf], 0.0, inplace=True)
    result.fillna(0.0, inplace=True)
    return result.astype(np.float32)

def sample_rows(frame: pd.DataFrame) -> pd.DataFrame:
    sampled = []
    for label_name, group in frame.groupby('label_name', sort=False):
        limit = BENIGN_MAX_ROWS if label_name == 'Benign' else ATTACK_MAX_ROWS_PER_CLASS
        sampled.append(group.sample(limit, random_state=RANDOM_STATE) if len(group) > limit else group)
    return pd.concat(sampled, ignore_index=True)

DATASET_PATH

PosixPath('/Users/minseok/Desktop/codex/프로젝트/DeepFence/training/data/88a47ba2ab64258e_MOHANAD_A4706/data/NF-CSE-CIC-IDS2018.csv')

In [2]:
raw = pd.read_csv(DATASET_PATH, usecols=[*BASE_FEATURES, 'Attack'])
raw['label_name'] = raw['Attack'].map(normalize_attack)
raw = sample_rows(raw)
raw['binary_label'] = (raw['label_name'] != 'Benign').astype(np.int64)
raw['attack_label'] = raw['label_name'].map(ATTACK_LABEL_MAPPING)

features = add_derived_features(raw[BASE_FEATURES])
print(raw.shape)
raw['label_name'].value_counts()

(1677791, 14)


label_name
Benign           1000000
Brute Force       200000
DDoS              200000
DoS               200000
Infiltration       62072
Bot                15683
SQL Injection         36
Name: count, dtype: int64

In [3]:
x_train_val, x_test, y_binary_train_val, y_binary_test, labels_train_val, labels_test = train_test_split(
    features[FEATURE_NAMES].to_numpy(dtype=np.float32),
    raw['binary_label'].to_numpy(dtype=np.int64),
    raw['label_name'].to_numpy(),
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=raw['label_name'],
)
val_ratio = VAL_SIZE / (1.0 - TEST_SIZE)
x_train, x_val, y_binary_train, y_binary_val, labels_train, labels_val = train_test_split(
    x_train_val,
    y_binary_train_val,
    labels_train_val,
    test_size=val_ratio,
    random_state=RANDOM_STATE,
    stratify=labels_train_val,
)

scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train).astype(np.float32)
x_val_scaled = scaler.transform(x_val).astype(np.float32)
x_test_scaled = scaler.transform(x_test).astype(np.float32)

attack_train_mask = labels_train != 'Benign'
attack_val_mask = labels_val != 'Benign'
attack_test_mask = labels_test != 'Benign'

y_attack_train = np.array([ATTACK_LABEL_MAPPING[label] for label in labels_train[attack_train_mask]], dtype=np.int64)
y_attack_val = np.array([ATTACK_LABEL_MAPPING[label] for label in labels_val[attack_val_mask]], dtype=np.int64)
y_attack_test = np.array([ATTACK_LABEL_MAPPING[label] for label in labels_test[attack_test_mask]], dtype=np.int64)

x_train_scaled.shape, x_val_scaled.shape, x_test_scaled.shape, y_attack_train.shape

((1174453, 17), (251669, 17), (251669, 17), (474453,))

In [4]:
np.save(OUTPUT_DIR / 'X_train.npy', x_train_scaled)
np.save(OUTPUT_DIR / 'X_val.npy', x_val_scaled)
np.save(OUTPUT_DIR / 'X_test.npy', x_test_scaled)
np.save(OUTPUT_DIR / 'y_binary_train.npy', y_binary_train)
np.save(OUTPUT_DIR / 'y_binary_val.npy', y_binary_val)
np.save(OUTPUT_DIR / 'y_binary_test.npy', y_binary_test)
np.save(OUTPUT_DIR / 'X_attack_train.npy', x_train_scaled[attack_train_mask])
np.save(OUTPUT_DIR / 'X_attack_val.npy', x_val_scaled[attack_val_mask])
np.save(OUTPUT_DIR / 'X_attack_test.npy', x_test_scaled[attack_test_mask])
np.save(OUTPUT_DIR / 'y_attack_train.npy', y_attack_train)
np.save(OUTPUT_DIR / 'y_attack_val.npy', y_attack_val)
np.save(OUTPUT_DIR / 'y_attack_test.npy', y_attack_test)

with (OUTPUT_DIR / 'scaler.pkl').open('wb') as file:
    pickle.dump(scaler, file)
(OUTPUT_DIR / 'feature_names.json').write_text(json.dumps(FEATURE_NAMES, ensure_ascii=False, indent=2), encoding='utf-8')
(OUTPUT_DIR / 'binary_label_mapping.json').write_text(json.dumps(BINARY_LABEL_MAPPING, ensure_ascii=False, indent=2), encoding='utf-8')
(OUTPUT_DIR / 'attack_label_mapping.json').write_text(json.dumps(ATTACK_LABEL_MAPPING, ensure_ascii=False, indent=2), encoding='utf-8')

meta = {
    'version': 'netflow_v2_twostage',
    'source': str(DATASET_PATH),
    'num_features': len(FEATURE_NAMES),
    'feature_names': FEATURE_NAMES,
    'binary_label_mapping': BINARY_LABEL_MAPPING,
    'attack_label_mapping': ATTACK_LABEL_MAPPING,
    'benign_max_rows': BENIGN_MAX_ROWS,
    'attack_max_rows_per_class': ATTACK_MAX_ROWS_PER_CLASS,
    'class_counts': raw['label_name'].value_counts().to_dict(),
    'splits': {
        'train': int(len(y_binary_train)),
        'val': int(len(y_binary_val)),
        'test': int(len(y_binary_test)),
        'attack_train': int(len(y_attack_train)),
        'attack_val': int(len(y_attack_val)),
        'attack_test': int(len(y_attack_test)),
    },
}
(OUTPUT_DIR / 'preprocess_meta.json').write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding='utf-8')
OUTPUT_DIR

PosixPath('/Users/minseok/Desktop/codex/프로젝트/DeepFence/training/data/processed_netflow_v2_twostage')